# HW04 Part A - Image Task: MNIST MLP vs CNN

이 노트북은 과제 제출용 baseline 코드이다. 위에서부터 순서대로 실행하면 데이터 로드, 모델 학습, 성능 저장, loss/accuracy 그래프 저장까지 수행한다.

## 실행 전 확인
- GPU 런타임 사용 권장
- 결과는 `./outputs` 폴더에 저장됨
- 최종 보고서에는 `*_metrics_summary.csv`, `*_loss_curve.png` 값을 반영하면 됨

In [ ]:
"""
Week 10 HW04 - Part A. Image Task
Dataset: MNIST (default) or CIFAR-10
Models: MLP baseline vs CNN baseline
Outputs:
  - outputs/taskA_metrics_summary.csv
  - outputs/taskA_history.csv
  - outputs/taskA_loss_curve.png
  - outputs/taskA_accuracy_curve.png

Run:
  python HW04_TaskA_Image_MNIST_MLP_CNN.py

Note:
  - Default is MNIST for fast homework execution.
  - To use CIFAR-10, change DATASET_NAME = "CIFAR10".
"""

import os
import random
from pathlib import Path
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


# =========================================================
# 0. Config
# =========================================================
SEED = 42
DATASET_NAME = "MNIST"  # "MNIST" or "CIFAR10"
BATCH_SIZE = 128
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
VALID_RATIO = 0.1
DATA_DIR = Path("./data")
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")


# =========================================================
# 1. Dataset
# =========================================================
def load_image_dataset(dataset_name: str):
    dataset_name = dataset_name.upper()

    if dataset_name == "MNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        train_full = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
        test_set = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)
        num_classes = 10

    elif dataset_name == "CIFAR10":
        transform_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
        train_full = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform_train)
        test_set = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_test)
        num_classes = 10

    else:
        raise ValueError("DATASET_NAME must be 'MNIST' or 'CIFAR10'.")

    val_size = int(len(train_full) * VALID_RATIO)
    train_size = len(train_full) - val_size
    train_set, val_set = random_split(
        train_full,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    sample_x, _ = train_full[0]
    input_shape = tuple(sample_x.shape)  # (C, H, W)

    return train_set, val_set, test_set, input_shape, num_classes


train_set, val_set, test_set, INPUT_SHAPE, NUM_CLASSES = load_image_dataset(DATASET_NAME)
print(f"[INFO] Dataset: {DATASET_NAME}")
print(f"[INFO] Input shape: {INPUT_SHAPE}, classes: {NUM_CLASSES}")
print(f"[INFO] Train/Val/Test: {len(train_set)}/{len(val_set)}/{len(test_set)}")

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# =========================================================
# 2. Models
# =========================================================
class MLPBaseline(nn.Module):
    def __init__(self, input_shape: Tuple[int, int, int], num_classes: int):
        super().__init__()
        c, h, w = input_shape
        input_dim = c * h * w
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNNBaseline(nn.Module):
    def __init__(self, input_shape: Tuple[int, int, int], num_classes: int):
        super().__init__()
        in_channels = input_shape[0]
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# =========================================================
# 3. Train / Evaluation functions
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def run_experiment(model_name: str, model: nn.Module) -> Tuple[pd.DataFrame, Dict[str, float]]:
    print(f"\n========== Training {model_name} ==========")
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = []
    best_val_acc = 0.0
    best_path = OUT_DIR / f"taskA_{model_name}_best.pt"

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        row = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train loss {train_loss:.4f}, acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f}, acc {val_acc:.4f}"
        )

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    summary = {
        "model": model_name,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    }
    print(f"[TEST] {model_name}: loss={test_loss:.4f}, acc={test_acc:.4f}")

    return pd.DataFrame(history), summary


# =========================================================
# 4. Run MLP and CNN
# =========================================================
mlp_history, mlp_summary = run_experiment("MLP", MLPBaseline(INPUT_SHAPE, NUM_CLASSES))
cnn_history, cnn_summary = run_experiment("CNN", CNNBaseline(INPUT_SHAPE, NUM_CLASSES))

history_df = pd.concat([mlp_history, cnn_history], ignore_index=True)
summary_df = pd.DataFrame([mlp_summary, cnn_summary])

history_df.to_csv(OUT_DIR / "taskA_history.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv(OUT_DIR / "taskA_metrics_summary.csv", index=False, encoding="utf-8-sig")
print("\n[INFO] Saved metrics CSV files in ./outputs")
print(summary_df)


# =========================================================
# 5. Plot loss and accuracy curves
# =========================================================
def plot_curves(history: pd.DataFrame, metric: str, ylabel: str, out_path: Path):
    plt.figure(figsize=(8, 5))
    for model_name in history["model"].unique():
        sub = history[history["model"] == model_name]
        plt.plot(sub["epoch"], sub[f"train_{metric}"], marker="o", label=f"{model_name} train")
        plt.plot(sub["epoch"], sub[f"val_{metric}"], marker="s", linestyle="--", label=f"{model_name} val")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"Part A Image Task - {ylabel} Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


plot_curves(history_df, "loss", "Loss", OUT_DIR / "taskA_loss_curve.png")
plot_curves(history_df, "acc", "Accuracy", OUT_DIR / "taskA_accuracy_curve.png")
print("[INFO] Saved plots: taskA_loss_curve.png, taskA_accuracy_curve.png")
